# Predict repeat orders with AdventureWorks on Databricks

Which customers who ordered during the previous year will order again in the next 90 days? This notebook trains a model from four AdventureWorks tables in Unity Catalog, writes predictions to a dedicated Databricks table, and retains a model bundle and prediction file in a Unity Catalog volume.

Complete [Databricks Notebook Setup](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/databricks-notebook) and [Connect to Unity Catalog](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/connect-your-data#databricks-unity-catalog) first. The tables `Customer`, `Product`, `SalesOrderHeader`, and `SalesOrderDetail` must already exist in the connector's catalog and schema. This notebook does not import the source data. Review the [Microsoft AdventureWorks samples](https://github.com/microsoft/sql-server-samples/tree/master/samples/databases/adventure-works) and their license before using the data.

Replace the angle bracket placeholders for your Databricks workspace, secret scope, SQL warehouse, catalog, schema, and artifact volume. Run the cells in order. Training and graph materialization use the notebook compute; the SQL warehouse reads the source tables and receives predictions. This notebook has no saved results.

## Install Kumo in this notebook

Follow the runtime-specific preparation in [Databricks Notebook Setup](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/databricks-notebook#install-kumo) before this installation cell. Then restart Python and continue with the imports and storage configuration below. Use GPU notebook compute with local SSD scratch.

In [ ]:
%pip install "kumo-engine[databricks]"

<br>

## Restart Python

The restart clears Python variables. Continue with the import and configuration cells afterward.

In [ ]:
dbutils.library.restartPython()

<br>

## Import Kumo and configure storage

Local scratch holds staged data, graph indexes, training files, and prediction files. Choose a directory backed by supported local storage with enough free space for your graph and working files. The Unity Catalog volume retains the completed artifacts when the cluster stops. A writable path alone does not establish its storage type or capacity.

In [ ]:
import kumo.engine as kumo
from pathlib import Path
import shutil
import tempfile

print("Kumo version:", kumo.__version__)

SCRATCH_DIR = Path("/local_disk0/kumo-scratch")
ARTIFACT_ROOT = Path("/Volumes/<catalog>/<schema>/<volume>/runs")

SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(dir=SCRATCH_DIR) as probe:
    probe.write(b"scratch check")
    probe.flush()
with tempfile.NamedTemporaryFile(dir=ARTIFACT_ROOT) as probe:
    probe.write(b"volume check")

kumo.configure(scratch_dir=str(SCRATCH_DIR))
print("Free scratch space (GiB):", round(shutil.disk_usage(SCRATCH_DIR).free / 2**30, 1))
print("Retained artifact root:", ARTIFACT_ROOT)

<br>

## Connect to Unity Catalog

Use an existing service principal with permission to read the four tables and write to a separate prediction destination. Keep its secret in a Databricks secret scope. The notebook user's volume access and the connector's SQL permissions are separate.

In [ ]:
identity = kumo.DatabricksIdentity(
    host="https://<workspace-host>",
    client_id=dbutils.secrets.get("<secret-scope>", "<client-id-key>"),
    client_secret=dbutils.secrets.get("<secret-scope>", "<client-secret-key>"),
)

connector = kumo.DatabricksConnector(
    identity=identity,
    warehouse_id="<sql-warehouse-id>",
    catalog="<catalog>",
    schema_name="<schema>",
)

<br>

## Check the source tables

Confirm the four tables are visible and inspect their columns before building the graph. Your extract must contain the columns used below. A schema check does not prove a full snapshot will fit or that the output table can be written.

In [ ]:
TABLE_NAMES = ["Customer", "Product", "SalesOrderHeader", "SalesOrderDetail"]
available = {name.lower() for name in connector.table_names()}
missing = [name for name in TABLE_NAMES if name.lower() not in available]
if missing:
    raise ValueError(f"Missing AdventureWorks tables in this schema: {missing}")

for name in TABLE_NAMES:
    source = connector[name]
    print(name, [(column.name, column.dtype, column.stype) for column in source.columns])

<br>

## Define the AdventureWorks tables

A graph describes which table keys connect and which columns Kumo can use. `CustomerID`, `ProductID`, `SalesOrderID`, and `SalesOrderDetailID` identify records. `OrderDate` supplies the order time. This example uses `ModifiedDate` as the detail time; check that it reflects when those details became available in your extract.

In [ ]:
from kumoapi.typing import Dtype, Stype

customer = (
    kumo.Table.from_source_table(
        connector["Customer"],
        ["CustomerID", "StoreID", "TerritoryID"],
        primary_key="CustomerID",
    )
    .with_column("StoreID", stype=Stype.categorical)
    .with_column("TerritoryID", stype=Stype.categorical)
)

product = (
    kumo.Table.from_source_table(
        connector["Product"],
        ["ProductID", "Name", "Color", "ProductLine", "Class",
         "StandardCost", "ListPrice"],
        primary_key="ProductID",
    )
    .with_column("Name", stype=Stype.text)
    .with_column("Color", stype=Stype.categorical)
    .with_column("ProductLine", stype=Stype.categorical)
    .with_column("Class", stype=Stype.categorical)
    .with_column("StandardCost", stype=Stype.numerical)
    .with_column("ListPrice", stype=Stype.numerical)
)

In [ ]:
sales_order_header = (
    kumo.Table.from_source_table(
        connector["SalesOrderHeader"],
        ["SalesOrderID", "CustomerID", "OrderDate", "OnlineOrderFlag",
         "TerritoryID", "SubTotal", "TotalDue"],
        primary_key="SalesOrderID",
        time_column="OrderDate",
    )
    .with_column("CustomerID", stype=Stype.ID)
    .with_column("OrderDate", dtype=Dtype.date)
    .with_column("OnlineOrderFlag", stype=Stype.categorical)
    .with_column("TerritoryID", stype=Stype.categorical)
    .with_column("SubTotal", stype=Stype.numerical)
    .with_column("TotalDue", stype=Stype.numerical)
)

sales_order_detail = (
    kumo.Table.from_source_table(
        connector["SalesOrderDetail"],
        ["SalesOrderDetailID", "SalesOrderID", "ProductID", "ModifiedDate",
         "OrderQty", "UnitPrice", "LineTotal"],
        primary_key="SalesOrderDetailID",
        time_column="ModifiedDate",
    )
    .with_column("SalesOrderID", stype=Stype.ID)
    .with_column("ProductID", stype=Stype.ID)
    .with_column("ModifiedDate", dtype=Dtype.date)
    .with_column("OrderQty", stype=Stype.numerical)
    .with_column("UnitPrice", stype=Stype.numerical)
    .with_column("LineTotal", stype=Stype.numerical)
)

<br>

## Build the graph and snapshot

Each edge connects a foreign key to its parent table. Validation checks the definition; the snapshot then reads the source data and creates local graph files. Keep those files available through training and prediction.

In [ ]:
from kumoapi.graph import Edge

graph = kumo.Graph(
    tables=[customer, product, sales_order_header, sales_order_detail],
    edges=[
        Edge(src_table="SalesOrderHeader", fkey="CustomerID", dst_table="Customer"),
        Edge(src_table="SalesOrderDetail", fkey="SalesOrderID", dst_table="SalesOrderHeader"),
        Edge(src_table="SalesOrderDetail", fkey="ProductID", dst_table="Product"),
    ],
)
graph.validate()

<br>

## Inspect the graph

Inspect the actual graph’s tables and relationships before materialization. The [graph visualization](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs) call is a commented preview because it is unavailable in the supported API.

In [ ]:
# graph.visualize(show_cols=True)

In [ ]:
snapshot = graph.snapshot()
print("Snapshot:", snapshot.id)

<br>

## Ask the prediction question

A predictive query defines the outcome and the customers to score. Here, customers with an order in the previous 365 days are selected, and the model predicts whether each places an order in the next 90 days.

In [ ]:
QUERY = """
PREDICT COUNT(SalesOrderHeader.*, 0, 90, days) > 0
FOR EACH Customer.CustomerID
WHERE COUNT(SalesOrderHeader.*, -365, 0, days) > 0
""".strip()

pquery = kumo.PredictiveQuery(graph=graph, query=QUERY)
pquery.validate()
print("Task:", pquery.get_task_type())

<br>

## Generate historical training rows

Use past orders as context and the next 90 days as the label window. The suggested temporal split evaluates later windows; inspect the generated table and plan before fitting.

In [ ]:
from kumoapi.model_plan import RunMode

train_table = pquery.generate_training_table(snapshot)
print("Training table:", train_table.train_table_path)


<br>

## Configure the model

Use FAST tutorial defaults and inspect the temporal split and validation metric.


In [ ]:
model_plan = pquery.suggest_model_plan(snapshot, run_mode=RunMode.FAST)
trainer = kumo.Trainer(model_plan=model_plan)
print(model_plan)


<br>

## Train

Fit the repeat-order classifier on the generated historical rows.


In [ ]:
fit_result = trainer.fit(train_table, snapshot=snapshot)


<br>

## Evaluate

Review validation and test metrics separately. AUROC measures how well repeat-order customers rank above non-ordering customers; precision and recall describe decisions at a chosen threshold.


In [ ]:
print("Metrics:", fit_result.metrics)
print("Local model bundle:", fit_result.bundle_dir)


<br>

## Batch predictions

Choose a prediction date suitable for your extract. June 30, 2025 follows the last order date in the linked Microsoft sample; change it if your prepared tables cover different dates. The query selects customers with an order during the preceding year. A date without future observations can still produce predictions, but it cannot supply outcomes for evaluating those predictions.

The output uses `WriteMode.OVERWRITE`, which replaces the contents of the dedicated output table. Change `PREDICTION_OUTPUT_NAME` to retain an earlier batch. Never use a source table as the output destination. The service principal also needs permission to use the connector's managed staging volume.

In [ ]:
from uuid import uuid4

from datetime import datetime
from kumoapi.jobs import PredictionArtifactType, WriteMode

PREDICTION_TIME = datetime(2025, 6, 30)
PREDICTION_OUTPUT_NAME = ("customer_order_predictions") + "_" + uuid4().hex[:8]
PREDICTION_TABLE = (f"{connector.schema_name}__kumo__{PREDICTION_OUTPUT_NAME}") + "_" + uuid4().hex[:8]

prediction_table = pquery.generate_prediction_table(
    snapshot,
    prediction_time=PREDICTION_TIME,
    plan=trainer.training_table_generation_plan,
)


<br>

### Score the customers

Run the classifier and write to the unique dedicated prediction table.


In [ ]:
pred_result = trainer.predict(
    prediction_table,
    snapshot=snapshot,
    output_config=kumo.PredictionOutputConfig(
        connector=connector,
        artifact_type=PredictionArtifactType.PREDICTIONS,
        table_name=PREDICTION_TABLE,
        write_mode=WriteMode.OVERWRITE,
    ),
    num_workers=1,
    binary_classification_threshold=0.5,
)


<br>

## Inspect batch results

Expect one row per eligible customer. `True_PROB` is the 90-day repeat-order probability and `TARGET_PRED` is the decision above 0.5. Verify customer IDs before saving the batch.


In [ ]:
import pandas as pd

population = pd.read_parquet(prediction_table.table_path)
assert len(population) >= 2, "Choose at least two scoring rows."
scoring_rows = population[[name for name in ("ENTITY1", "TIMESTAMP") if name in population]]
assert not any(str(column).startswith("TARGET") for column in scoring_rows)
batch_scores = pred_result.frame
assert set(batch_scores["ENTITY"]) == set(population["ENTITY1"])
if str(pquery.get_task_type()) in {"binary_classification", "regression"}:
    assert len(batch_scores) == len(population)
print(batch_scores.head(10).to_string(index=False))

scores = batch_scores


<br>

## Retain the model and prediction file

The model bundle and local prediction files disappear when the driver storage is removed. Copy the complete model bundle and a Parquet copy of predictions to the configured Unity Catalog volume before stopping compute. `pred_result.frame` loads all scored rows into pandas; this simple copy suits a small example, but larger results need a file based transfer.

In [ ]:
from uuid import uuid4

RUN_ROOT = ARTIFACT_ROOT / uuid4().hex
RUN_ROOT.mkdir(parents=True, exist_ok=False)
shutil.copytree(Path(fit_result.bundle_dir), RUN_ROOT / "model")
scores.to_parquet(RUN_ROOT / "predictions.parquet", index=False)
print("Record this run path:", RUN_ROOT)

<br>

## Reuse the model later

After a cluster restart, reinstall a compatible Kumo build, configure new scratch, reconnect to the four tables, and rebuild a compatible graph and snapshot. Load the complete saved bundle with `kumo.Trainer.load()` instead of generating a new training table or fitting again. The bundle does not include the source tables or graph index. See [Publish and Retrieve Models](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/publish-and-retrieve-models#retrieve-a-published-model) for the reload procedure.